# Berliner Straßennetz — Exploration

Loads the Berlin street segment data (WFS download stored locally), dissolves by street name,
calculates lengths and derives key statistics.

**Data source:** Straßenabschnitte RBS, Senatsverwaltung für Stadtentwicklung via gdi.berlin.de  
**License:** CC-BY-3.0 © Amt für Statistik Berlin-Brandenburg  
**Key fields:** `strname` (street name), `bezname` (Bezirk), `strnr` (street ID), `strabtypname` (type)

All length calculations use **EPSG:25833** (UTM Zone 33N — Berlin's native metric CRS).

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Load the local WFS download (WGS84), reproject to metric CRS for length calculations
strassen_raw = gpd.read_file('../storytelling/data/strassen_berlin_wgs84.geojson')
strassen_raw = strassen_raw.to_crs(epsg=25833)

print(f'{len(strassen_raw):,} Straßensegmente')
print(f'CRS: {strassen_raw.crs}')
strassen_raw.head(3)

In [ ]:
print('Spalten:', strassen_raw.columns.tolist())
print('Straßentypen:')
strassen_raw['strabtypname'].value_counts()

## Segmente nach Straßenname zusammenführen

In [ ]:
# Dissolve by street name — one row per unique strname
strassen = strassen_raw.dissolve(by='strname', aggfunc={
    'strnr': 'first',
    'bezname': lambda x: ', '.join(sorted(set(x))),
    'strabtypname': 'first'
}).reset_index()

strassen['laenge'] = strassen.geometry.length
strassen['n_bezirke'] = strassen_raw.groupby('strname')['bezname'].nunique().reindex(strassen['strname']).values

print(f'{len(strassen):,} eindeutige Straßennamen')
strassen.sort_values('laenge', ascending=False).head()

## Längste Straßen

Autobahnen und Wasserstraßen werden herausgefiltert — sie sind im RBS als Verkehrsobjekte enthalten, aber keine Straßen im eigentlichen Sinne.

In [ ]:
EXCLUDE = ['BAB', 'Wasserstraße', 'kanal', 'Kanal', 'Havel', 'Spree', 'Wuhle',
           'Panke', 'Dahme', 'Nordgraben', 'Teltow', 'BLW', 'Verbindungsweg']

proper = strassen[~strassen['strname'].str.contains('|'.join(EXCLUDE), na=False)].copy()

top20 = proper.nlargest(20, 'laenge')[['strname', 'laenge', 'n_bezirke', 'bezname']].copy()
top20['laenge_km'] = (top20['laenge'] / 1000).round(2)
top20[['strname', 'laenge_km', 'n_bezirke', 'bezname']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top10 = top20.head(10)
ax.barh(top10['strname'], top10['laenge_km'], color='#e74c3c')
ax.set_xlabel('Länge (km)')
ax.set_title('Top 10 längste Straßen Berlins')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Kürzeste Straßen

In [ ]:
bottom20 = proper.nsmallest(20, 'laenge')[['strname', 'laenge', 'bezname']].copy()
bottom20['laenge_m'] = bottom20['laenge'].round(1)
bottom20[['strname', 'laenge_m', 'bezname']]

## Straßen, die die meisten Bezirke verbinden

In [ ]:
multi_bez = proper[proper['n_bezirke'] >= 4].sort_values('n_bezirke', ascending=False)
multi_bez[['strname', 'laenge', 'n_bezirke', 'bezname']].assign(
    laenge_km=lambda df: (df['laenge']/1000).round(2)
)[['strname', 'laenge_km', 'n_bezirke', 'bezname']]

## Straßentypen: Was Berlin von anderen Städten unterscheidet

Berlin hat **Chausseen**, **Alleen** und **Damme** in einer Dichte, die andere deutsche Städte nicht kennen —
ein Erbe der preußischen Stadtplanung und der Eingemeindung umliegender Gemeinden 1920.

In [ ]:
def classify_type(name):
    name = str(name)
    if name.endswith(('straße', 'strasse', 'Str.', 'str.')): return 'Straße'
    if 'allee' in name.lower(): return 'Allee'
    if name.endswith(('weg', 'Weg')): return 'Weg'
    if name.endswith(('platz', 'Platz', 'Pl.')): return 'Platz'
    if 'damm' in name.lower(): return 'Damm'
    if 'chaussee' in name.lower(): return 'Chaussee'
    if 'allee' in name.lower(): return 'Allee'
    if name.endswith(('ring', 'Ring')): return 'Ring'
    if name.startswith(('Am ', 'An ', 'Auf ', 'Im ', 'In ', 'Zum ', 'Zur ')): return 'Präposition'
    if name.startswith('Alt-') or name.startswith('Neu-'): return 'Alt/Neu'
    return 'Andere'

proper['typ'] = proper['strname'].apply(classify_type)
type_counts = proper['typ'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
type_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Straßentypen in Berlin')
ax.set_xlabel('')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(type_counts)

## Berliner Eigenheiten: Dörfer in der Großstadt

Berlin ist 1920 durch die Eingemeindung von 7 Städten, 59 Landgemeinden und 27 Gutsbezirken entstanden.
Spuren davon finden sich in Straßennamen mit `Alt-`, `Dorf` und `Kietz`.

In [ ]:
# Village remnants
dorf_mask = strassen_raw['strname'].str.contains('Alt-|Dorfstr|Am Dorf|An der Dorf|Kietz|Dorfanger|Dorfkirche', na=False)
dorf_streets = strassen_raw[dorf_mask][['strname', 'bezname', 'ortname']].drop_duplicates(subset='strname')
print(f'{len(dorf_streets)} Straßennamen mit Dorf-/Kietz-Bezug:')
dorf_streets.sort_values('strname')

## Längste Straßennamen

In [ ]:
name_lengths = strassen_raw[['strname']].drop_duplicates()
name_lengths['nchar'] = name_lengths['strname'].str.len()

# Exclude administrative/BLW entries
real_names = name_lengths[~name_lengths['strname'].str.startswith(('BLW', 'Verbindungsweg'), na=False)]
real_names.nlargest(15, 'nchar')[['strname', 'nchar']]

## Straßennamen nach Bezirk

In [ ]:
seg_per_bez = strassen_raw.groupby('bezname').agg(
    n_segmente=('gisid', 'count'),
    n_unique_names=('strname', 'nunique')
).sort_values('n_segmente', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
seg_per_bez['n_unique_names'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Eindeutige Straßennamen pro Bezirk')
ax.set_xlabel('Anzahl')
plt.tight_layout()
plt.show()

print(seg_per_bez)

## Ortsteil-Analyse: Wie die Dörfer noch in Berlin stecken

Berlin hat 96 Ortsteile (Stadtteile innerhalb der 12 Bezirke). Viele Ortsteile tragen noch die Namen
ihrer ursprünglichen Dörfer.

In [ ]:
orte = strassen_raw.groupby(['bezname', 'ortname']).size().reset_index(name='n_segmente')
print(f'{orte["ortname"].nunique()} Ortsteile in {orte["bezname"].nunique()} Bezirken')
orte.groupby('bezname')['ortname'].count().sort_values(ascending=False)

## Karte: Alle Straßen

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12), facecolor='#1a1a1a')
strassen_raw.plot(ax=ax, linewidth=0.1, color='#e0e0e0', alpha=0.5)
ax.set_title('Berliner Straßennetz', color='white', fontsize=16)
ax.axis('off')
plt.tight_layout()
plt.show()